In [1]:
import os
# Java 8 path
os.environ['JAVA_HOME'] = '/Library/Java/JavaVirtualMachines/temurin-8.jdk/Contents/Home'

# PySpark path
os.environ['SPARK_HOME'] = '/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/pyspark'

In [22]:
import numpy as np
from scipy import stats
from pyspark.ml.feature import *
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd 


In [23]:
from pyspark.sql import SparkSession
import re 
spark = SparkSession.builder \
    .appName("Worksheet significance") \
    .getOrCreate() 
sc = spark.sparkContext

In [25]:
state_df = spark.read.csv(
    "/Users/dikshanta/Documents/Softwarica 2nd sem Fils/BigData/state.csv",
    header=True,
    inferSchema=True
)

state_df.show(5)


+----------+------+----------+--------+------+-------+-----+------+
|Population|Income|Illiteracy|Life Exp|Murder|HS Grad|Frost|  Area|
+----------+------+----------+--------+------+-------+-----+------+
|      3615|  3624|       2.1|   69.05|  15.1|   41.3|   20| 50708|
|       365|  6315|       1.5|   69.31|  11.3|   66.7|  152|566432|
|      2212|  4530|       1.8|   70.55|   7.8|   58.1|   15|113417|
|      2110|  3378|       1.9|   70.66|  10.1|   39.9|   65| 51945|
|     21198|  5114|       1.1|   71.71|  10.3|   62.6|   20|156361|
+----------+------+----------+--------+------+-------+-----+------+
only showing top 5 rows



In [26]:
state_pd = state_df.toPandas()
# to know the value dim - in r in 
state_pd.head(5)

,Population,Income,Illiteracy,Life Exp,Murder,HS Grad,Frost,Area
0,3615,3624,2.1,69.05,15.1,41.3,20,50708
1,365,6315,1.5,69.31,11.3,66.7,152,566432
2,2212,4530,1.8,70.55,7.8,58.1,15,113417
3,2110,3378,1.9,70.66,10.1,39.9,65,51945
4,21198,5114,1.1,71.71,10.3,62.6,20,156361


In [27]:
state_pd.shape


(50, 8)

## Hypothesis Testing

In [28]:
state_pd.describe()

,Population,Income,Illiteracy,Life Exp,Murder,HS Grad,Frost,Area
count,50.000000,50.000000,50.000000,50.000000,50.00000,50.000000,50.000000,50.000000
mean,4246.420000,4435.800000,1.170000,70.878600,7.37800,53.108000,104.460000,70735.880000
std,4464.491433,614.469939,0.609533,1.342394,3.69154,8.076998,51.980848,85327.299622
min,365.000000,3098.000000,0.500000,67.960000,1.40000,37.800000,0.000000,1049.000000
25%,1079.500000,3992.750000,0.625000,70.117500,4.35000,48.050000,66.250000,36985.250000
50%,2838.500000,4519.000000,0.950000,70.675000,6.85000,53.250000,114.500000,54277.000000
75%,4968.500000,4813.500000,1.575000,71.892500,10.67500,59.150000,139.750000,81162.500000
max,21198.000000,6315.000000,2.800000,73.600000,15.10000,67.300000,188.000000,566432.000000


In [42]:
murder_data = state_pd['Murder'].values
print(murder_data)

[15.1 11.3  7.8 10.1 10.3  6.8  3.1  6.2 10.7 13.9  6.2  5.3 10.3  7.1
  2.3  4.5 10.6 13.2  2.7  8.5  3.3 11.1  2.3 12.5  9.3  5.   2.9 11.5
  3.3  5.2  9.7 10.9 11.1  1.4  7.4  6.4  4.2  6.1  2.4 11.6  1.7 11.
 12.2  4.5  5.5  9.5  4.3  6.7  3.   6.9]


In [51]:
np.mean(murder_data)

7.377999999999999

In [63]:
t_stat = (np.mean(murder_data)-8.5)/(np.std(murder_data, ddof = 1))/(np.sqrt(len(murder_data)))

In [64]:
df = len(murder_data)

In [65]:
p_value = 2 * stats.t.sf(np.abs(t_stat),df)

In [72]:
ci = stats.t.interval(
    0.95,
    df=df,
    loc=np.mean(murder_data),
    scale=np.std(murder_data, ddof=1) / np.sqrt(len(murder_data))
)


In [73]:
print("Mean:", np.mean(murder_data))
print("Standard Deviation:", np.std(murder_data, ddof=1))


Mean: 7.377999999999999
Standard Deviation: 3.69153969315277


In [82]:
print("test stats:" , t_stat)

test stats: -0.0429833551546468


## Test Statistics 

t_stat = **2.149**  

This means:  
- The sample mean is **3.149 standard errors** away from **8.5**  
- **Negative sign** means the sample mean is below **8.5**  
- The larger the **|t|**, the more **unusual** our sample is  

- **|t| = 0.5**  ----> very close to **hypothesized mean** (not unusual)  
- **|t| = 1.5**  ----> moderately far from **hypothesized mean**  
- **|t| = 2.1**  ----> far from **hypothesized mean**  
